# Notebook to re-create the Figures and Table from Section 3. Results

To create the figures in this notebook, the trained LIF-DL needs to have been deployed to create a forecast, and evaluations need to have been performed.

See scripts/ forecast.py and evaluate.py. Remember to save the intermediate results of the evaluation by setting the --save_intermediate flag in the command line.

In [ ]:
# IMPORTS
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import os

# Import plotting functions from src.utils
from src.utils import (
    plot_variable_importance_grouped,
    plot_spatial_timing_maps,
    plot_local_morans_i,
    plot_fic_temporal_evolution
)

In [ ]:
## SETUP - set constants for the notebook

# Define model path
model_name = "LIF_DL_Best"
eval_dir = Path(f"../results/{model_name}/evaluations")

# Obs order
obs_order = ["IMS", "CIS"]
# Model order
model_order = ["LIF_DL", "FLake"]
# Lake order
lake_order = [
    "Great_Bear_Lake",
    "Great_Slave_Lake",
    "Lake_Athabasca",
    "Reindeer_Lake",
    "Lake_Winnipeg"
]

# Set the figure path if you want to save the figures
fig_dir = Path(f"../results/{model_name}/figures/")
# fig_path = None
if fig_dir and not fig_dir.exists():
    os.mkdir(fig_dir)

# Check if the evaluation directory exists and has the desired files
if not eval_dir.exists():
    raise FileNotFoundError(f"Evaluation directory {eval_dir} does not exist.")

# check for fic, overall, phenology and spatial csvs
required_files = [
    "fic.csv",
    "overall.csv",
    "phenology.csv",
    "spatial.csv"
]
for file in required_files:
    if not (eval_dir / file).exists():
        raise FileNotFoundError(f"Required file {file} not found in {eval_dir}.")
    
# Check for intermediate files
intermediate_dir = eval_dir / "intermediate"
if not intermediate_dir.exists():
    raise FileNotFoundError(f"Intermediate directory {intermediate_dir} does not exist.")


## 3.1 Variable Importance

In [ ]:
# Load variable importance results
vi_df = pd.read_csv(eval_dir / "variable_importance.csv")

print(f"Loaded variable importance from: {eval_dir / 'variable_importance.csv'}")
print(f"Shape: {vi_df.shape}")
print()
print("Variable Importance Results:")
print("="*70)
print(vi_df.round(4).to_string(index=False))
print("="*70)

In [ ]:
# Sort variables by average importance across both freezeup and breakup
sorted_vars = vi_df.set_index('Variable')[['Freezeup', 'Breakup']].mean(axis=1)
sorted_vars = sorted_vars.sort_values(ascending=False).index.tolist()
custom_variable_order = sorted_vars

# Define custom labels for cleaner display names
custom_variable_labels = {
    'temperature_2m': 'Air Temperature',
    'wind_speed_10m': 'Wind Speed',
    'surface_solar_radiation_downwards_sum': 'Solar Radiation',
    'total_precipitation_sum': 'Precipitation',
    'relative_humidity': 'Relative Humidity',
    'total_cloud_cover': 'Cloud Cover',
    'accumulated_freezing_dd': 'AFDD',
    'accumulated_thawing_dd': 'ATDD',
    'lake_depth': 'Lake Depth'
}

# Create the grouped bar chart
fig = plot_variable_importance_grouped(
    vi_df, 
    variable_order=custom_variable_order,
    variable_labels=custom_variable_labels,
    save_path=fig_dir/"variable_importance.png"
)
plt.show()

## 3.2 Full Forecast Evaluation

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set display options for pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.float_format', '{:.4f}'.format)

In [ ]:
# Load overall evaluation CSV
overall_df = pd.read_csv(eval_dir / "overall.csv")

# Compute "Overall" results (average across lakes) per Obs and Model without attempting to average the Lake column
# Keep only the Obs/Model and numeric columns for aggregation
cols_for_agg = [c for c in overall_df.columns if c not in ['Lake']]
average_df = overall_df[cols_for_agg].groupby(['Obs', 'Model'], as_index=False).mean()
average_df['Lake'] = 'Average'

# Reorder columns to match original: Lake, Obs, Model, then the rest
rest_cols = [c for c in overall_df.columns if c not in ['Lake', 'Obs', 'Model']]
average_df = average_df[['Lake', 'Obs', 'Model'] + rest_cols]

# Append the overall rows to the phenology dataframe
overall_df = pd.concat([overall_df, average_df], ignore_index=True)

print(f"Loaded overall evaluation from: {eval_dir / 'overall.csv'}")
print(f"Shape: {overall_df.shape}")
print()

# Convert Lake names to match the order (fix capitalization)
overall_df['Lake'] = overall_df['Lake'].str.replace('_', ' ').str.title().str.replace(' ', '_')
overall_df['Model'] = overall_df['Model'].str.upper().replace('LIF-DL', 'LIF_DL').replace('FLAKE', 'FLake')

# Create categorical ordering for sorting
overall_df['Lake'] = pd.Categorical(overall_df['Lake'], categories=lake_order+["Average"], ordered=True)
overall_df['Obs'] = pd.Categorical(overall_df['Obs'], categories=obs_order, ordered=True)
overall_df['Model'] = pd.Categorical(overall_df['Model'], categories=model_order, ordered=True)

# Sort by Lake, Obs, Model
overall_df = overall_df.sort_values(['Lake', 'Obs', 'Model'])

# Set multi-index
display_table = overall_df.set_index(['Lake', 'Obs', 'Model'])

# Display the full table
print("Full Forecast Evaluation - Overall Metrics")
print("="*120)

# Set the pandas float format for precision
pd.options.display.float_format = '{:.2f}'.format
display_table

## 3.3 Spatial Accuracy

In [ ]:
# Load spatial map data from intermediate files
import xarray as xr

# Path to intermediate spatial maps
spatial_dir = eval_dir / "intermediate" / "spatial_maps"

# Load all spatial maps for each lake and source
spatial_data = {}
sources = ["ims", "lif_dl", "flake"]

for lake in lake_order:
    lake_key = lake.lower().replace(' ', '_')
    spatial_data[lake] = {}
    
    for source in sources:
        file_path = spatial_dir / f"{lake_key}_{source}.nc"
        if file_path.exists():
            ds = xr.open_dataset(file_path)
            spatial_data[lake][source] = ds
            print(f"Loaded: {file_path.name}")
        else:
            print(f"Missing: {file_path.name}")

print(f"\nLoaded spatial data for {len(spatial_data)} lakes")
print(f"Sources per lake: {sources}")

In [ ]:
# Load the spatial evaluation metrics for SSIM values
spatial_df = pd.read_csv(eval_dir / "spatial.csv")

# Fix lake and model names to match
spatial_df['Lake'] = spatial_df['Lake'].str.replace('_', ' ').str.title().str.replace(' ', '_')
spatial_df['Model'] = spatial_df['Model'].str.upper().replace('LIF-DL', 'LIF-DL')

print(f"Loaded spatial metrics: {spatial_df.shape}")
spatial_df.head()

### 3.3.1 Break-up Start

In [ ]:
# Create Break-up Start (BUS) spatial timing anomaly map
fig_bus = plot_spatial_timing_maps(spatial_data, season='BUS', spatial_df=spatial_df, lake_order=lake_order, save_path=fig_dir/"BUS_timing.png")
plt.show()

In [ ]:
# Create Break-up Start (BUS) Local Moran's I cluster map
fig_bus_lmi = plot_local_morans_i(spatial_data, season='BUS', spatial_df=spatial_df, lake_order=lake_order, save_path=fig_dir/"BUS_lmi.png")
plt.show()

### 3.3.2 Freeze-up Start

In [ ]:
# Create Freeze-up Start (FUS) spatial timing anomaly map
fig_fus = plot_spatial_timing_maps(spatial_data, season='FUS', spatial_df=spatial_df, lake_order=lake_order, save_path=fig_dir/"FUS_timing.png")
plt.show()

In [ ]:
# Create Freeze-up Start (FUS) Local Moran's I cluster map
fig_fus_lmi = plot_local_morans_i(spatial_data, season='FUS', spatial_df=spatial_df, lake_order=lake_order, save_path=fig_dir/"FUS_lmi.png")
plt.show()

## 3.4 Temporal Evolution of Lake Ice Fraction

In [ ]:
# Load FIC timeseries data from intermediate files
fic_dir = eval_dir / "intermediate" / "fic_timeseries"

# Load all FIC timeseries for each lake and source
fic_data = {}
fic_sources = ["ims", "cis", "lif_dl", "flake"]

for lake in lake_order:
    lake_key = lake.lower().replace(' ', '_')
    fic_data[lake] = {}
    
    for source in fic_sources:
        file_path = fic_dir / f"{lake_key}_{source}.csv"
        if file_path.exists():
            df = pd.read_csv(file_path, parse_dates=['Date'])
            fic_data[lake][source] = df
            print(f"Loaded: {file_path.name}")
        else:
            print(f"Missing: {file_path.name}")

print(f"\nLoaded FIC data for {len(fic_data)} lakes")
print(f"Sources available: {fic_sources}")

In [ ]:
# Create FIC temporal evolution plot
fig_fic = plot_fic_temporal_evolution(fic_data, lake_order, save_path=fig_dir/"fic_trend.png")
plt.show()

In [ ]:
## Display the results table for the fraction of ice cover (FIC) evaluation
fic_df = pd.read_csv(eval_dir / "fic.csv")
# Drop n_samples column if exists
if 'n_samples' in fic_df.columns:
    fic_df = fic_df.drop(columns=['n_samples'])

# Compute the Average across lakes, grouped by Obs and Model
cols_for_agg = [col for col in fic_df.columns if col not in ['Lake']]
avg_fic_df = fic_df[cols_for_agg].groupby(['Obs', 'Model'], as_index=False).mean()
avg_fic_df['Lake'] = 'Average'

# Reorder columns to match original
rest_cols = [col for col in fic_df.columns if col not in ['Lake']]
avg_fic_df = avg_fic_df[['Lake'] + rest_cols]

# Append the Average row to the original fic_df
fic_df = pd.concat([fic_df, avg_fic_df], ignore_index=True)

print("Fraction of Ice Cover (FIC) Evaluation Metrics")
print("="*100)

# Create the display table with multi-index
fic_df['Lake'] = fic_df['Lake'].str.replace('_', ' ').str.title().str.replace(' ', '_')
fic_df['Model'] = fic_df['Model'].str.upper().replace('LIF-DL', 'LIF_DL').replace('FLAKE', 'FLake')
fic_df['Lake'] = pd.Categorical(fic_df['Lake'], categories=lake_order + ['Average'], ordered=True)
fic_df['Obs'] = pd.Categorical(fic_df['Obs'], categories=obs_order, ordered=True)
fic_df['Model'] = pd.Categorical(fic_df['Model'], categories=model_order, ordered=True)
fic_df = fic_df.sort_values(['Lake', 'Obs', 'Model'])
display_fic_table = fic_df.set_index(['Lake', 'Obs', 'Model'])

# Set pandas float display precision to 2 decimals
pd.options.display.float_format = '{:.2f}'.format

display_fic_table

## 3.5 Temporal Accuracy of Ice Phenology Events

In [ ]:
# Load phenology evaluation CSV
phenology_df = pd.read_csv(eval_dir / "phenology.csv")

# Compute "Overall" results (average across lakes) per Obs and Model without attempting to average the Lake column
# Keep only the Obs/Model and numeric columns for aggregation
cols_for_agg = [c for c in phenology_df.columns if c not in ['Lake']]
overall_phenology = phenology_df[cols_for_agg].groupby(['Obs', 'Model'], as_index=False).mean()
overall_phenology['Lake'] = 'Average'

# Reorder columns to match original: Lake, Obs, Model, then the rest
rest_cols = [c for c in phenology_df.columns if c not in ['Lake', 'Obs', 'Model']]
overall_phenology = overall_phenology[['Lake', 'Obs', 'Model'] + rest_cols]

# Append the overall rows to the phenology dataframe
phenology_df = pd.concat([phenology_df, overall_phenology], ignore_index=True)

print(f"Loaded phenology evaluation from: {eval_dir / 'phenology.csv'}")
print(f"Shape: {phenology_df.shape}")
print()

# Similar to the overall display table, create a phenology display table
phenology_df['Lake'] = phenology_df['Lake'].str.replace('_', ' ').str.title().str.replace(' ', '_')
phenology_df['Model'] = phenology_df['Model'].str.upper().replace('LIF-DL', 'LIF_DL').replace('FLAKE', 'FLake')

# Sort based on the pre-defined order
phenology_df['Lake'] = pd.Categorical(phenology_df['Lake'], categories=lake_order+["Average"], ordered=True)
phenology_df['Obs'] = pd.Categorical(phenology_df['Obs'], categories=obs_order, ordered=True)
phenology_df['Model'] = pd.Categorical(phenology_df['Model'], categories=model_order, ordered=True)

# Sort by Lake, Obs, Model
phenology_df = phenology_df.sort_values(['Lake', 'Obs', 'Model'])

# Set multi-index
display_table = phenology_df.set_index(['Lake', 'Obs', 'Model'])

# Display the full table
print("Full Forecast Evaluation - phenology Metrics")
print("="*120)


# Set pandas float display precision to 2 decimals
pd.options.display.float_format = '{:.0f}'.format

display_table